# AHN Compressive Memory — RQ2 Pilot

**Project:** What Does the Artificial Hippocampus Store?  
**Model:** Qwen2.5-3B-Instruct + AHN-GatedDeltaNet  
**Sliding window:** 128 tokens (shrunk for tractability)  
**Goal:** Test whether AHN's compressed memory retains evicted content, using logit lens as a preliminary readout (J-lens blocked by version conflict).

## Structure
1. Setup and load model
2. Install hooks (module-level, safe pattern)
3. Sanity check hooks fire correctly
4. Layer profile — which layers are AHN-active
5. Multi-needle Δ-readout — 3 needles across all layers
6. Retention decay curve — vary eviction distance
7. Save all results

**Design decisions:**
- Hooks captured in a `Captures` object (never wiped by other cells)
- Δ-readout (AHN-on minus NOWRITE) to isolate AHN's contribution from base priors
- All figures saved to disk

## 1. Setup and load model

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from ahn.transformer.qwen2_ahn import register_customized_qwen2

register_customized_qwen2()
MODEL_PATH = "/workspace/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN"
RESULTS_DIR = "/workspace/AHN/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map="cuda",
    attn_implementation="flash_attention_2",
)
model.eval()
unembed = model.lm_head.weight  # [vocab_size, hidden_dim]
VOCAB_SIZE, HIDDEN = unembed.shape
SLIDING_WINDOW = model.config.sliding_window
N_LAYERS = model.config.num_hidden_layers

print(f"Model:  Qwen2.5-3B + AHN-GDN")
print(f"Vocab:  {VOCAB_SIZE}, Hidden: {HIDDEN}, Layers: {N_LAYERS}")
print(f"Window: {SLIDING_WINDOW}, Use sliding: {model.config.use_sliding_window}")

## 2. Install hooks (safe pattern)

Hooks are managed by a class so they persist across cells and don't accidentally get wiped. Every AHN forward call writes into `captures.data`.

In [ ]:
class AHNCaptureManager:
    """Persistent capture of AHN outputs across all layers."""
    def __init__(self, model):
        self.data = {}
        self.hook_handles = []
        self.ahn_layers = [i for i, l in enumerate(model.model.layers) if hasattr(l, 'ahn')]
        self._install(model)
    
    def _make_hook(self, idx):
        def hook(module, input, output):
            out = output[0] if isinstance(output, tuple) else output
            self.data[idx] = out.detach().clone()
        return hook
    
    def _install(self, model):
        # Wipe any existing hooks on ahn modules
        for layer in model.model.layers:
            if hasattr(layer, 'ahn'):
                layer.ahn._forward_hooks.clear()
        # Install fresh
        for i, layer in enumerate(model.model.layers):
            if hasattr(layer, 'ahn'):
                h = layer.ahn.register_forward_hook(self._make_hook(i))
                self.hook_handles.append(h)
    
    def clear_data(self):
        self.data.clear()
    
    def get_ot(self, layer_idx):
        """Return o_t at the last sequence position for a given layer."""
        if layer_idx not in self.data:
            raise KeyError(f"Layer {layer_idx} not captured. Did AHN activate? "
                          f"Prompt must exceed {SLIDING_WINDOW} tokens.")
        t = self.data[layer_idx].squeeze()
        if t.dim() > 1:
            t = t[-1]
        return t

def zero_ahn_hook(module, input, output):
    """Hook that zeros AHN output — for NOWRITE control."""
    if isinstance(output, tuple):
        return (torch.zeros_like(output[0]),) + output[1:]
    return torch.zeros_like(output)

captures = AHNCaptureManager(model)
print(f"Installed {len(captures.hook_handles)} capture hooks")

## 3. Sanity check — hooks fire and AHN activates

Verifies:
- Prompt exceeds sliding window
- All layers capture o_t
- o_t has correct shape (batch=1, hidden_dim)

In [ ]:
FILLER_UNIT = "The quick brown fox jumps. "
test_prompt = f"The special word is 42. {FILLER_UNIT * 100} What was the special word?"
test_inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
n_toks = test_inputs['input_ids'].shape[1]

captures.clear_data()
with torch.no_grad():
    model(**test_inputs, use_cache=True)

sample = captures.get_ot(N_LAYERS // 2)
print(f"Prompt tokens:      {n_toks}")
print(f"Sliding window:     {SLIDING_WINDOW}")
print(f"AHN should be active: {n_toks > SLIDING_WINDOW}")
print(f"Layers captured:    {len(captures.data)}/{N_LAYERS}")
print(f"Sample o_t shape:   {captures.data[N_LAYERS//2].shape}")
print(f"Sample o_t dtype:   {sample.dtype}")
assert len(captures.data) == N_LAYERS, "Not all layers captured!"
assert sample.shape == (HIDDEN,), f"Wrong shape: {sample.shape}"
print("\n[OK] Sanity check passed.")

## 4. Layer profile — which layers are AHN-active?

For each layer, compare o_t (AHN on) vs o_t (AHN zeroed). Layers where these differ substantially are the ones where AHN is contributing.

In [ ]:
# Run 1: AHN on
captures.clear_data()
with torch.no_grad():
    model(**test_inputs, use_cache=True)
ahn_snapshot = {k: v.clone() for k, v in captures.data.items()}

# Run 2: AHN zeroed
zh = [l.ahn.register_forward_hook(zero_ahn_hook) for l in model.model.layers if hasattr(l, 'ahn')]
captures.clear_data()
with torch.no_grad():
    model(**test_inputs, use_cache=True)
nw_snapshot = {k: v.clone() for k, v in captures.data.items()}
for h in zh: h.remove()

# Compute AHN contribution % per layer
contributions = {}
for i in sorted(ahn_snapshot.keys()):
    a = ahn_snapshot[i].squeeze()
    n = nw_snapshot[i].squeeze()
    if a.dim() > 1: a = a[-1]
    if n.dim() > 1: n = n[-1]
    diff = (a - n).abs().sum().item()
    norm = max(a.abs().sum().item(), 1e-9)
    contributions[i] = 100 * diff / norm

active_layers = [i for i, c in contributions.items() if c > 20]
print(f"Active layers (>20% AHN contribution): {len(active_layers)}/{N_LAYERS}")
print(f"Range: {active_layers[0]}-{active_layers[-1]}")

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
layers = list(contributions.keys())
pcts = list(contributions.values())
colors = ['crimson' if p > 20 else 'lightgray' for p in pcts]
ax.bar(layers, pcts, color=colors)
ax.axhline(y=20, color='k', linestyle='--', alpha=0.5, label='20% threshold')
ax.set_xlabel('Layer index')
ax.set_ylabel('AHN contribution (%)')
ax.set_title(f'AHN activity by layer (Qwen2.5-3B + AHN-GDN)')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/layer_profile.png', dpi=120)
plt.show()

with open(f'{RESULTS_DIR}/layer_profile.json', 'w') as f:
    json.dump(contributions, f, indent=2)

## 5. Multi-needle Δ-readout across all layers

For each needle, at each layer, compute Δ-readout (AHN − NOWRITE) and rank the needle in vocabulary space. Low rank = strong retention.

In [ ]:
NEEDLES = ["Paris", "42", "banana"]
FILLER_100 = FILLER_UNIT * 100  # ~600 tokens, ensures eviction

def measure_delta_readout(prompt, needle_id):
    """Return dict {layer_idx: rank_of_needle} using Δ-readout."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    captures.clear_data()
    with torch.no_grad():
        model(**inputs, use_cache=True)
    ahn_snap = {k: v.clone() for k, v in captures.data.items()}
    
    zh = [l.ahn.register_forward_hook(zero_ahn_hook) for l in model.model.layers if hasattr(l, 'ahn')]
    captures.clear_data()
    with torch.no_grad():
        model(**inputs, use_cache=True)
    nw_snap = {k: v.clone() for k, v in captures.data.items()}
    for h in zh: h.remove()
    
    ranks = {}
    for i in ahn_snap:
        a, n = ahn_snap[i].squeeze(), nw_snap[i].squeeze()
        if a.dim() > 1: a = a[-1]
        if n.dim() > 1: n = n[-1]
        delta = a - n
        logits = delta @ unembed.T
        rank = (logits.argsort(descending=True) == needle_id).nonzero()[0].item()
        ranks[i] = rank
    return ranks

delta_results = {}
for needle in NEEDLES:
    needle_id = tokenizer.encode(f" {needle}", add_special_tokens=False)[0]
    prompt = f"The special word is {needle}. {FILLER_100} What was the special word?"
    ranks = measure_delta_readout(prompt, needle_id)
    delta_results[needle] = ranks
    vals = list(ranks.values())
    print(f"{needle:8s}: min={min(vals):6d}  median={sorted(vals)[len(vals)//2]:6d}  best_layer={min(ranks, key=ranks.get)}")

# Plot: 3 needles × 36 layers
fig, ax = plt.subplots(figsize=(11, 5))
for needle, ranks in delta_results.items():
    xs = sorted(ranks.keys())
    ys = [ranks[x] for x in xs]
    ax.plot(xs, ys, marker='o', label=needle, alpha=0.8)
ax.axhline(y=VOCAB_SIZE//2, color='gray', linestyle='--', alpha=0.5, label='Random (median)')
ax.set_yscale('log')
ax.set_xlabel('Layer index')
ax.set_ylabel('Needle rank (lower = better retention)')
ax.set_title(f'Δ-readout across layers, 3 needles (Qwen2.5-3B + AHN-GDN, window={SLIDING_WINDOW})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/multi_needle_delta.png', dpi=120)
plt.show()

with open(f'{RESULTS_DIR}/multi_needle_delta.json', 'w') as f:
    json.dump({k: {str(i): r for i, r in v.items()} for k, v in delta_results.items()}, f, indent=2)

## 6. Retention decay curve — vary eviction distance

Fix the needle at position 0, vary the filler length. Longer filler = larger eviction distance. Track needle rank at each needle's best layer.

In [ ]:
# Best layer per needle from previous cell
BEST_LAYERS = {n: min(delta_results[n], key=delta_results[n].get) for n in NEEDLES}
FILLER_COUNTS = [25, 50, 100, 150, 200, 300]

print("Best layers from previous cell:", BEST_LAYERS)

decay_curves = {}
for needle in NEEDLES:
    needle_id = tokenizer.encode(f" {needle}", add_special_tokens=False)[0]
    best_layer = BEST_LAYERS[needle]
    curve = {}
    for n_fill in FILLER_COUNTS:
        prompt = f"The special word is {needle}. " + FILLER_UNIT * n_fill + "What was the special word?"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        n_toks = inputs['input_ids'].shape[1]
        eviction_dist = max(0, n_toks - SLIDING_WINDOW - 6)
        
        captures.clear_data()
        with torch.no_grad():
            model(**inputs, use_cache=True)
        a = captures.get_ot(best_layer)
        
        zh = [l.ahn.register_forward_hook(zero_ahn_hook) for l in model.model.layers if hasattr(l, 'ahn')]
        captures.clear_data()
        with torch.no_grad():
            model(**inputs, use_cache=True)
        n = captures.get_ot(best_layer)
        for h in zh: h.remove()
        
        delta = a - n
        rank = ((delta @ unembed.T).argsort(descending=True) == needle_id).nonzero()[0].item()
        curve[eviction_dist] = rank
    decay_curves[needle] = curve
    print(f"{needle}: {curve}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
for needle, curve in decay_curves.items():
    xs = sorted(curve.keys())
    ys = [curve[x] for x in xs]
    ax.plot(xs, ys, marker='o', linewidth=2, label=f"{needle} (layer {BEST_LAYERS[needle]})")
ax.axhline(y=VOCAB_SIZE//2, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_yscale('log')
ax.set_xlabel('Eviction distance (tokens past window)')
ax.set_ylabel('Needle rank (lower = better retention)')
ax.set_title(f'AHN retention decay — 3 needles at their best layers')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/retention_decay.png', dpi=120)
plt.show()

with open(f'{RESULTS_DIR}/retention_decay.json', 'w') as f:
    json.dump({k: {str(i): r for i, r in v.items()} for k, v in decay_curves.items()}, f, indent=2)

## 7. Summary

All results saved to `/workspace/AHN/results/`:
- `layer_profile.png`, `layer_profile.json`
- `multi_needle_delta.png`, `multi_needle_delta.json`
- `retention_decay.png`, `retention_decay.json`

**Next steps:**
1. Repeat on AHN-DeltaNet 3B (same code, change MODEL_PATH)
2. Repeat on AHN-Mamba2 3B
3. Compare retention curves across cells → RQ1 + RQ2
4. If time: try J-lens in a separate env

In [ ]:
print("=== Session summary ===")
print(f"Model:            Qwen2.5-3B + AHN-GatedDeltaNet")
print(f"Sliding window:   {SLIDING_WINDOW}")
print(f"AHN-active layers: {len(active_layers)}/{N_LAYERS}")
print()
print("Best (min) needle ranks:")
for needle in NEEDLES:
    r = min(delta_results[needle].values())
    print(f"  {needle:8s}: rank {r:6d}  at layer {BEST_LAYERS[needle]}")
print()
print("Retention holds beyond eviction:", any(
    min(v.values()) < 10000 for v in decay_curves.values()
))
print()
print(f"Files saved to: {RESULTS_DIR}")